In [1]:
import pandas as pd
import requests, io

In [2]:
api_url = "https://data.moenv.gov.tw/api/v2/aqx_p_02?api_key=af57253c-e838-46da-a1f5-12b43afd75f3&limit=1000&sort=datacreationdate%20desc&format=CSV"

In [26]:
api_json = "https://data.moenv.gov.tw/api/v2/aqx_p_02?api_key=846e44e1-8cc5-4893-ad87-c79d2d383706&limit=1000&sort=datacreationdate%20desc&format=JSON"

## 避開SSL認證

In [27]:
resp = requests.get(api_url, verify=False)
print(resp)
df = pd.read_csv(io.StringIO(resp.text))
df

<Response [200]>


c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'data.moenv.gov.tw'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,因受限於資源分配，每日呼叫API的次數不可大於5000次。


In [28]:
df = pd.read_json(api_json)
df

,site,county,pm25,datacreationdate,itemunit
0,林森,臺南市,18,2026-04-19 13:00,μg/m3
1,員林,彰化縣,20,2026-04-19 13:00,μg/m3
2,臺灣大道,臺中市,13,2026-04-19 13:00,μg/m3
3,大城,彰化縣,19,2026-04-19 13:00,μg/m3
4,富貴角,新北市,8,2026-04-19 13:00,μg/m3
...,...,...,...,...,...
995,線西,彰化縣,24,2026-04-19 01:00,μg/m3
996,彰化,彰化縣,24,2026-04-19 01:00,μg/m3
997,西屯,臺中市,24,2026-04-19 01:00,μg/m3
998,忠明,臺中市,23,2026-04-19 01:00,μg/m3


In [4]:
df = pd.read_csv(api_url)
df

,因受限於資源分配，每日呼叫API的次數不可大於5000次。


## 清理資料、移除空值、移除重複

In [29]:
## 查看資料結構→ 空值、筆數、型態
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   site              1000 non-null   str  
 1   county            1000 non-null   str  
 2   pm25              1000 non-null   int64
 3   datacreationdate  1000 non-null   str  
 4   itemunit          1000 non-null   str  
dtypes: int64(1), str(4)
memory usage: 39.2 KB


In [30]:
df.describe()

,pm25
count,1000.000000
mean,21.721000
std,8.429129
min,0.000000
25%,15.000000
50%,21.000000
75%,28.000000
max,46.000000


In [31]:
# 查看重複值
# df.duplicated()
# df[df.duplicated(subset=["site", "datacreationdate"])]

# df.drop_duplicates()
df.drop_duplicates(subset=["site", "datacreationdate"])

,site,county,pm25,datacreationdate,itemunit
0,林森,臺南市,18,2026-04-19 13:00,μg/m3
1,員林,彰化縣,20,2026-04-19 13:00,μg/m3
2,臺灣大道,臺中市,13,2026-04-19 13:00,μg/m3
3,大城,彰化縣,19,2026-04-19 13:00,μg/m3
4,富貴角,新北市,8,2026-04-19 13:00,μg/m3
...,...,...,...,...,...
995,線西,彰化縣,24,2026-04-19 01:00,μg/m3
996,彰化,彰化縣,24,2026-04-19 01:00,μg/m3
997,西屯,臺中市,24,2026-04-19 01:00,μg/m3
998,忠明,臺中市,23,2026-04-19 01:00,μg/m3


### .dropna() 移除空值

In [8]:
df.drop_duplicates(subset=["site", "datacreationdate"]).dropna()

,因受限於資源分配，每日呼叫API的次數不可大於5000次。


In [32]:
df1 = df.drop_duplicates(subset=["site", "datacreationdate"]).dropna()
df1

,site,county,pm25,datacreationdate,itemunit
0,林森,臺南市,18,2026-04-19 13:00,μg/m3
1,員林,彰化縣,20,2026-04-19 13:00,μg/m3
2,臺灣大道,臺中市,13,2026-04-19 13:00,μg/m3
3,大城,彰化縣,19,2026-04-19 13:00,μg/m3
4,富貴角,新北市,8,2026-04-19 13:00,μg/m3
...,...,...,...,...,...
995,線西,彰化縣,24,2026-04-19 01:00,μg/m3
996,彰化,彰化縣,24,2026-04-19 01:00,μg/m3
997,西屯,臺中市,24,2026-04-19 01:00,μg/m3
998,忠明,臺中市,23,2026-04-19 01:00,μg/m3


## sqlite 建立資料庫
- unique(site, datacreationdate)
    - 插入資料唯一的約束
- 整數寫法→ integer
    - 不能寫 int
- autoincrement
    - 不是寫 auto_increment

In [10]:
import sqlite3

In [11]:
sqlstr = '''
create table if not exists data(
id integer primary key autoincrement,
site text,
county text,
pm25 integer,
datacreationdate text,
itemunit text,
unique(site, datacreationdate)
)
'''

In [12]:
conn = sqlite3.connect("pm25.db")
cursor = conn.cursor()
conn, cursor

(<sqlite3.Connection at 0x22936d6c040>, <sqlite3.Cursor at 0x22936d54d40>)

In [13]:
cursor.execute(sqlstr)
conn.commit()

### 插入資料
- or ignore
    - 忽略重複資料


In [33]:
sqlstr = "insert or ignore into data (site,county,pm25,datacreationdate,itemunit)\
    values(?, ?, ?, ?, ?)"

In [34]:
# df1.values
df1.values.tolist()

[['林森', '臺南市', 18, '2026-04-19 13:00', 'μg/m3'],
 ['員林', '彰化縣', 20, '2026-04-19 13:00', 'μg/m3'],
 ['臺灣大道', '臺中市', 13, '2026-04-19 13:00', 'μg/m3'],
 ['大城', '彰化縣', 19, '2026-04-19 13:00', 'μg/m3'],
 ['富貴角', '新北市', 8, '2026-04-19 13:00', 'μg/m3'],
 ['麥寮', '雲林縣', 12, '2026-04-19 13:00', 'μg/m3'],
 ['關山', '臺東縣', 13, '2026-04-19 13:00', 'μg/m3'],
 ['馬公', '澎湖縣', 10, '2026-04-19 13:00', 'μg/m3'],
 ['金門', '金門縣', 23, '2026-04-19 13:00', 'μg/m3'],
 ['馬祖', '連江縣', 26, '2026-04-19 13:00', 'μg/m3'],
 ['埔里', '南投縣', 19, '2026-04-19 13:00', 'μg/m3'],
 ['復興', '高雄市', 21, '2026-04-19 13:00', 'μg/m3'],
 ['永和', '新北市', 14, '2026-04-19 13:00', 'μg/m3'],
 ['竹山', '南投縣', 26, '2026-04-19 13:00', 'μg/m3'],
 ['中壢', '桃園市', 15, '2026-04-19 13:00', 'μg/m3'],
 ['三重', '新北市', 13, '2026-04-19 13:00', 'μg/m3'],
 ['冬山', '宜蘭縣', 11, '2026-04-19 13:00', 'μg/m3'],
 ['宜蘭', '宜蘭縣', 13, '2026-04-19 13:00', 'μg/m3'],
 ['陽明', '臺北市', 17, '2026-04-19 13:00', 'μg/m3'],
 ['花蓮', '花蓮縣', 15, '2026-04-19 13:00', 'μg/m3'],
 ['臺東', '臺東縣', 17,

https://inloop.github.io/sqlite-viewer/

In [16]:
# 插入多筆 .executemany
cursor.executemany(sqlstr, df1.values.tolist())
conn.commit()

In [17]:
# cursor.rowcount 查看更新幾筆資料
cursor.rowcount

0

In [18]:
conn.close()